In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
csv_path = '/content/drive/MyDrive/2nd SeD/F4/data/주거용건물_최종좌표.csv'
df = pd.read_csv(csv_path)
display(df.head())

,도로명대지위치,주용도코드명,세대수,연면적(㎡),위도,경도,가중치,UTM_X,UTM_Y
0,경상북도 영주시 영주로231번길 20 (영주동),단독주택,1,629.97,36.826526,128.625504,0.786950,1.100372e+06,1.870405e+06
1,경상북도 영주시 영주로231번길 27 (영주동),단독주택,1,326.01,36.826960,128.625070,0.786950,1.100333e+06,1.870452e+06
2,경상북도 영주시 영주로231번길 27 (영주동),공동주택,0,66.08,36.826960,128.625070,1.797266,1.100333e+06,1.870452e+06
3,경상북도 영주시 영주로231번길 24 (영주동),단독주택,1,501.57,36.826726,128.625581,0.786950,1.100378e+06,1.870427e+06
4,경상북도 영주시 영주로231번길 30 (영주동),단독주택,1,401.64,36.826958,128.625607,0.786950,1.100380e+06,1.870453e+06


In [4]:
import folium
import branca.colormap as cm

# Filter out buildings with a weight of 0 (if not already done)
df_filtered = df[df['가중치'] > 0].reset_index(drop=True)

print(f"Original number of buildings: {len(df)}")
print(f"Number of buildings after filtering zero-weight: {len(df_filtered)}")


Original number of buildings: 23918
Number of buildings after filtering zero-weight: 12578


In [5]:
# Calculate the mean latitude and longitude for centering the map
mean_lat = df_filtered['위도'].mean()
mean_lon = df_filtered['경도'].mean()

# Create a base map centered at the mean coordinates
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=13)

# Create a colormap for weights (reds)
# The scale will go from the minimum non-zero weight to the maximum weight
min_weight_val = df_filtered['가중치'].min()
max_weight_val = df_filtered['가중치'].max()

# Ensure min_weight_val is not zero for the colormap, if all filtered weights are >0
if min_weight_val == 0 and len(df_filtered[df_filtered['가중치'] > 0]) > 0:
    min_weight_val = df_filtered[df_filtered['가중치'] > 0]['가중치'].min()

colormap = cm.LinearColormap(['#fee0d2', '#fc9272', '#de2d26'],
                                index=[min_weight_val, (min_weight_val + max_weight_val) / 2, max_weight_val],
                                caption='Building Weight (가중치)')

m.add_child(colormap)

# Add markers for each building
for idx, row in df_filtered.iterrows():
    # Get color from colormap based on weight
    color = colormap(row['가중치'])

    # Create popup content
    popup_content = f"<b>도로명대지위치:</b> {row['도로명대지위치']}<br><b>가중치:</b> {row['가중치']:.2f}"

    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=3, # Small radius for individual buildings
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=popup_content
    ).add_to(m)

# Display the map
display(m)


Output hidden; open in https://colab.research.google.com to view.


**클러스터 생성 조건**
1.  **클러스터 개수 (K):** 50개
2.  **클러스터 반경 (R):** 클러스터의 모든 건물은 중심으로부터 800m 이내에 있어야 함.
3.  **허용수치 반경 (r) 및 비율:** 클러스터 내 80% 이상의 건물이 중심으로부터 700m 이내에 있어야 함.
4.  **최소 가중치 합 (W):** 클러스터 내 건물들의 가중치 합은 최소 200 이상이어야 함.


+ 추가 : 가중치가 0인 건물은 애초에 고려 하지 않음

In [6]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances

# Define clustering parameters (already set in kernel state, but explicitly re-stating for clarity)
MAX_CLUSTERS_K = 50
RADIUS_R_METERS = 800 # All buildings must be within R meters from the center
RADIUS_r_METERS = 700 # 80% of buildings must be within r meters from the center
PERCENTAGE_r = 0.8 # Changed to 0.7 (70%)
MIN_TOTAL_WEIGHT_W = 200 # As specified by user for their previous successful model

print(f"Applying K-Means with the following parameters:")
print(f"K: {MAX_CLUSTERS_K}, R: {RADIUS_R_METERS}m, r: {RADIUS_r_METERS}m, Percentage_r: {PERCENTAGE_r*100}%, Min_Total_Weight: {MIN_TOTAL_WEIGHT_W}")


# Prepare data for K-Means
X = df_filtered[['UTM_X', 'UTM_Y']].values
sample_weights = df_filtered['가중치'].values

# Step 1: Initial K-Means Clustering with sample_weight
# Using n_init='auto' for robustness and better centroid initialization
kmeans = KMeans(n_clusters=MAX_CLUSTERS_K, random_state=42, n_init='auto')
df_filtered_kmeans = df_filtered.copy() # Work on a copy of the filtered dataframe
df_filtered_kmeans['cluster_label'] = kmeans.fit_predict(X, sample_weight=sample_weights)
centroids = kmeans.cluster_centers_

print(f"Initial K-Means clustering completed. {MAX_CLUSTERS_K} clusters formed before validation.")

final_identified_clusters = []
# assigned_buildings_original_indices = set() # Condition 5 removed: A building can belong to multiple clusters

for i in range(MAX_CLUSTERS_K):
    current_cluster_members_df = df_filtered_kmeans[df_filtered_kmeans['cluster_label'] == i].copy()

    if current_cluster_members_df.empty:
        continue

    cluster_centroid = centroids[i]

    # Calculate distances from current centroid to all its members
    member_coords = current_cluster_members_df[['UTM_X', 'UTM_Y']].values
    distances = euclidean_distances(member_coords, cluster_centroid.reshape(1, -1)).flatten()
    current_cluster_members_df['distance_to_centroid'] = distances

    # Step 2: Apply Filtering Logic based on user's previous model

    # Condition 1: R meters (800m) exclusion - filter out buildings outside R
    members_within_R = current_cluster_members_df[
        current_cluster_members_df['distance_to_centroid'] <= RADIUS_R_METERS
    ].copy()

    if members_within_R.empty:
        # print(f"  Cluster {i} failed: No members within R ({RADIUS_R_METERS}m).")
        continue

    # Condition 2: r meters (700m) percentage check
    members_within_r = members_within_R[
        members_within_R['distance_to_centroid'] <= RADIUS_r_METERS
    ]

    if len(members_within_R) > 0 and (len(members_within_r) / len(members_within_R)) < PERCENTAGE_r:
        # print(f"  Cluster {i} failed: Less than {PERCENTAGE_r*100}% of buildings within r ({RADIUS_r_METERS}m).")
        continue

    # Identify members for this cluster (exclusivity condition removed)
    # Now all valid members within R are considered for this cluster
    valid_member_original_indices = members_within_R.index.tolist()

    if not valid_member_original_indices:
        continue

    # Condition 3: Minimum total weight check
    total_weight_of_valid_members = df_filtered.loc[valid_member_original_indices, '가중치'].sum()

    if total_weight_of_valid_members < MIN_TOTAL_WEIGHT_W:
        # print(f"  Cluster {i} failed: Total weight of valid members ({total_weight_of_valid_members:.2f}) < {MIN_TOTAL_WEIGHT_W}.")
        continue

    # If all conditions passed, this is a valid cluster
    final_identified_clusters.append({
        'cluster_id': len(final_identified_clusters) + 1, # Assign a new sequential ID for final clusters
        'centroid_utm': cluster_centroid.tolist(),
        'members_original_indices': valid_member_original_indices,
        'num_members': len(valid_member_original_indices),
        'total_weight': total_weight_of_valid_members
    })
    # assigned_buildings_original_indices.update(valid_member_original_indices) # Condition 5 removed

print(f"Validation complete. Total final identified clusters: {len(final_identified_clusters)}")

# Assign final cluster IDs to df_filtered
# Note: If a building belongs to multiple validated clusters, 'cluster_id_kmeans_validated' will store the ID of the last cluster it was assigned to
df_filtered['cluster_id_kmeans_validated'] = -1 # Initialize with -1 (unclustered)

for cluster_info in final_identified_clusters:
    cluster_id = cluster_info['cluster_id']
    for idx in cluster_info['members_original_indices']:
        df_filtered.loc[idx, 'cluster_id_kmeans_validated'] = cluster_id

Applying K-Means with the following parameters:
K: 50, R: 800m, r: 700m, Percentage_r: 80.0%, Min_Total_Weight: 200
Initial K-Means clustering completed. 50 clusters formed before validation.
Validation complete. Total final identified clusters: 16


In [7]:
print("\nSummary of Final K-Means Clusters after validation:")
if final_identified_clusters:
    for cluster in final_identified_clusters:
        # For printing, we can approximate Lat/Lon by finding a member close to the centroid
        # Or, ideally, use a UTM to Lat/Lon conversion if available.
        # For now, let's print the UTM centroid.
        print(f"Cluster {cluster['cluster_id']}:")
        print(f"  Centroid (UTM_X, UTM_Y): ({cluster['centroid_utm'][0]:.2f}, {cluster['centroid_utm'][1]:.2f})")
        print(f"  Number of Members: {cluster['num_members']}")
        print(f"  Total Weight: {cluster['total_weight']:.2f}")
else:
    print("No clusters were identified after validation with the given conditions.")

# Display a sample of df_filtered with the new cluster IDs
display(df_filtered.head())


Summary of Final K-Means Clusters after validation:
Cluster 1:
  Centroid (UTM_X, UTM_Y): (1099729.61, 1868554.31)
  Number of Members: 752
  Total Weight: 606.90
Cluster 2:
  Centroid (UTM_X, UTM_Y): (1091324.28, 1875509.78)
  Number of Members: 533
  Total Weight: 501.70
Cluster 3:
  Centroid (UTM_X, UTM_Y): (1100661.54, 1870060.66)
  Number of Members: 901
  Total Weight: 771.26
Cluster 4:
  Centroid (UTM_X, UTM_Y): (1098972.85, 1870821.23)
  Number of Members: 165
  Total Weight: 220.67
Cluster 5:
  Centroid (UTM_X, UTM_Y): (1101363.36, 1870104.86)
  Number of Members: 245
  Total Weight: 324.41
Cluster 6:
  Centroid (UTM_X, UTM_Y): (1098788.90, 1868811.31)
  Number of Members: 352
  Total Weight: 743.96
Cluster 7:
  Centroid (UTM_X, UTM_Y): (1100028.60, 1868165.03)
  Number of Members: 1030
  Total Weight: 704.86
Cluster 8:
  Centroid (UTM_X, UTM_Y): (1099889.24, 1869647.73)
  Number of Members: 504
  Total Weight: 442.91
Cluster 9:
  Centroid (UTM_X, UTM_Y): (1091416.42, 1874830

,도로명대지위치,주용도코드명,세대수,연면적(㎡),위도,경도,가중치,UTM_X,UTM_Y,cluster_id_kmeans_validated
0,경상북도 영주시 영주로231번길 20 (영주동),단독주택,1,629.97,36.826526,128.625504,0.786950,1.100372e+06,1.870405e+06,3
1,경상북도 영주시 영주로231번길 27 (영주동),단독주택,1,326.01,36.826960,128.625070,0.786950,1.100333e+06,1.870452e+06,11
2,경상북도 영주시 영주로231번길 27 (영주동),공동주택,0,66.08,36.826960,128.625070,1.797266,1.100333e+06,1.870452e+06,11
3,경상북도 영주시 영주로231번길 24 (영주동),단독주택,1,501.57,36.826726,128.625581,0.786950,1.100378e+06,1.870427e+06,3
4,경상북도 영주시 영주로231번길 30 (영주동),단독주택,1,401.64,36.826958,128.625607,0.786950,1.100380e+06,1.870453e+06,3


In [8]:
df_Shuttle = df_filtered[df_filtered['cluster_id_kmeans_validated'] != -1].copy()

print(f"Original df_filtered rows: {len(df_filtered)}")
print(f"df_Shuttle (clustered buildings only) rows: {len(df_Shuttle)}")

display(df_Shuttle.head())

Original df_filtered rows: 12578
df_Shuttle (clustered buildings only) rows: 9759


,도로명대지위치,주용도코드명,세대수,연면적(㎡),위도,경도,가중치,UTM_X,UTM_Y,cluster_id_kmeans_validated
0,경상북도 영주시 영주로231번길 20 (영주동),단독주택,1,629.97,36.826526,128.625504,0.786950,1.100372e+06,1.870405e+06,3
1,경상북도 영주시 영주로231번길 27 (영주동),단독주택,1,326.01,36.826960,128.625070,0.786950,1.100333e+06,1.870452e+06,11
2,경상북도 영주시 영주로231번길 27 (영주동),공동주택,0,66.08,36.826960,128.625070,1.797266,1.100333e+06,1.870452e+06,11
3,경상북도 영주시 영주로231번길 24 (영주동),단독주택,1,501.57,36.826726,128.625581,0.786950,1.100378e+06,1.870427e+06,3
4,경상북도 영주시 영주로231번길 30 (영주동),단독주택,1,401.64,36.826958,128.625607,0.786950,1.100380e+06,1.870453e+06,3


In [9]:
import pyproj

# Define the UTM projection for South Korea (typically EPSG:5179 for Korea 2000 / UTM-K Zone 51)
# This is a common CRS for UTM-K in South Korea
utm_proj = pyproj.Proj(init='epsg:5179')

# Define the WGS84 (Lat/Lon) projection
wgs84_proj = pyproj.Proj(init='epsg:4326')

def latlon_to_utm(lat, lon):
    """Converts WGS84 Latitude and Longitude to UTM-K (EPSG:5179) X and Y coordinates."""
    utm_x, utm_y = pyproj.transform(wgs84_proj, utm_proj, lon, lat)
    return utm_x, utm_y

print("UTM conversion function (latlon_to_utm) defined.")

UTM conversion function (latlon_to_utm) defined.


/usr/local/lib/python3.12/dist-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/usr/local/lib/python3.12/dist-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


###=====클러스터링 결과=====

In [11]:
import folium
import branca.colormap as cm

# Define color palette for clusters (can be extended or use a colormap for more clusters)
cluster_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
    '#ff9933', '#66cc99', '#ff6666', '#80b3ff', '#ffb380', # Additional colors
    '#ffff66', '#ccff66', '#66ffcc', '#66ccff', '#cc66ff', # Additional colors
    '#ff66ff', '#ffb3ff', '#b3ffb3', '#b3b3ff', '#ffb3b3'  # Additional colors
]

unclustered_color = '#d3d3d3' # Light gray for unclustered buildings
centroid_marker_color = '#000000' # Black for centroids

# Calculate the mean latitude and longitude for centering the map
mean_lat_overall = df_filtered['위도'].mean()
mean_lon_overall = df_filtered['경도'].mean()

# Create a base map centered at the mean coordinates
m_clusters = folium.Map(location=[mean_lat_overall, mean_lon_overall], zoom_start=13)

# Add individual buildings (clustered and unclustered)
for idx, row in df_filtered.iterrows():
    cluster_id = row['cluster_id_kmeans_validated']

    if cluster_id != -1:
        # Clustered building
        # Use modulo operator to cycle through cluster_colors if MAX_CLUSTERS_K > len(cluster_colors)
        color = cluster_colors[(int(cluster_id) - 1) % len(cluster_colors)]
        popup_content = f"<b>도로명대지위치:</b> {row['도로명대지위치']}<br><b>가중치:</b> {row['가중치']:.2f}<br><b>클러스터 ID:</b> {int(cluster_id)}"
    else:
        # Unclustered building
        color = unclustered_color
        popup_content = f"<b>도로명대지위치:</b> {row['도로명대지위치']}<br><b>가중치:</b> {row['가중치']:.2f}<br><b>클러스터 ID:</b> N/A (미분류)"

    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=2, # Smaller radius for individual buildings
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=popup_content
    ).add_to(m_clusters)


# Add cluster centroids
if final_identified_clusters:
    for cluster_info in final_identified_clusters:
        cluster_id = cluster_info['cluster_id']
        # Calculate mean lat/lon for the members of this specific cluster for centroid visualization
        member_indices = cluster_info['members_original_indices']
        if member_indices:
            centroid_lat = df_filtered.loc[member_indices, '위도'].mean()
            centroid_lon = df_filtered.loc[member_indices, '경도'].mean()
        else:
            # Fallback if no members (shouldn't happen with validated clusters)
            centroid_lat = mean_lat_overall
            centroid_lon = mean_lon_overall

        folium.Circle(
            location=[centroid_lat, centroid_lon],
            radius=150, # Larger radius to clearly show centroid (in meters on map)
            color=centroid_marker_color,
            fill=True,
            fill_color=centroid_marker_color,
            fill_opacity=0.2,
            popup=f"<b>클러스터 ID:</b> {cluster_id}<br><b>멤버 수:</b> {cluster_info['num_members']}<br><b>총 가중치:</b> {cluster_info['total_weight']:.2f}"
        ).add_to(m_clusters)

# Add the user-defined polygon
polygon_coords = [
    [36.7798, 128.6197],
    [36.7751, 128.6167],
    [36.7808, 128.6076],
    [36.7895, 128.6181],
    [36.7847, 128.6248]
]

folium.Polygon(
    locations=polygon_coords,
    color='blue',
    fill=True,
    fill_color='blue',
    fill_opacity=0.3,
    tooltip='사용자 정의 영역'
).add_to(m_clusters)


# Display the map
display(m_clusters)

Output hidden; open in https://colab.research.google.com to view.

###=====셔틀 정류장 위치 도출(16개)=====
네이버지도 스트리트 뷰를 이용한 가상 분석으로 직접 도출

In [14]:
# Define the shuttle station points
Shuttle_station = [
 [36.8104, 128.6283],
[36.8178, 128.6276],
[36.8231, 128.6280],
[36.8233, 128.6356],
[36.8274, 128.6311],
[36.8272, 128.6194],
[36.8197, 128.6199],
[36.8146, 128.6190],
[36.8102, 128.6177],
[36.8075, 128.6267],
[36.8109, 128.6078],
[36.8183, 128.6046],
[36.8308, 128.6078],
[36.8674, 128.5253],
[36.8693, 128.5323],
[36.8731, 128.5242]
]

In [15]:
# Convert Shuttle_station points to UTM coordinates
shuttle_station_utm = []
for i, (lat, lon) in enumerate(Shuttle_station):
    utm_x, utm_y = latlon_to_utm(lat, lon)
    shuttle_station_utm.append({
        'id': i + 1,
        'lat': lat,
        'lon': lon,
        'utm_x': utm_x,
        'utm_y': utm_y
    })

# Display the converted points
import pandas as pd
df_shuttle_station_coords = pd.DataFrame(shuttle_station_utm)
display(df_shuttle_station_coords.head())

/tmp/ipykernel_35302/3115151846.py:12: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  utm_x, utm_y = pyproj.transform(wgs84_proj, utm_proj, lon, lat)


,id,lat,lon,utm_x,utm_y
0,1,36.8104,128.6283,1.100642e+06,1.868619e+06
1,2,36.8178,128.6276,1.100570e+06,1.869439e+06
2,3,36.8231,128.6280,1.100599e+06,1.870027e+06
3,4,36.8233,128.6356,1.101276e+06,1.870057e+06
4,5,36.8274,128.6311,1.100870e+06,1.870507e+06


In [12]:
import folium
import branca.colormap as cm

# Define color palette for clusters
cluster_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
    '#ff9933', '#66cc99', '#ff6666', '#80b3ff', '#ffb380',
    '#ffff66', '#ccff66', '#66ffcc', '#66ccff', '#cc66ff',
    '#ff66ff', '#ffb3ff', '#b3ffb3', '#b3b3ff', '#ffb3b3'
]

centroid_marker_color = '#000000' # Black for centroids

# Calculate the mean latitude and longitude for centering the map, using df_Shuttle as it's the data we are visualizing
mean_lat_shuttle = df_Shuttle['위도'].mean()
mean_lon_shuttle = df_Shuttle['경도'].mean()

# Create a base map centered at the mean coordinates of the clustered buildings
m_shuttle_clusters = folium.Map(location=[mean_lat_shuttle, mean_lon_shuttle], zoom_start=13)

# Add individual clustered buildings from df_Shuttle
for idx, row in df_Shuttle.iterrows():
    cluster_id = row['cluster_id_kmeans_validated']

    # Use modulo operator to cycle through cluster_colors
    color = cluster_colors[(int(cluster_id) - 1) % len(cluster_colors)]
    popup_content = f"<b>도로명대지위치:</b> {row['도로명대지위치']}<br><b>가중치:</b> {row['가중치']:.2f}<br><b>클러스터 ID:</b> {int(cluster_id)}"

    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=2, # Smaller radius for individual buildings
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=popup_content
    ).add_to(m_shuttle_clusters)


# User-provided 16 points to add
Shuttle_station = [
 [36.8104, 128.6283],
[36.8178, 128.6276],
[36.8231, 128.6280],
[36.8233, 128.6356],
[36.8274, 128.6311],
[36.8272, 128.6194],
[36.8197, 128.6199],
[36.8146, 128.6190],
[36.8102, 128.6177],
[36.8075, 128.6267],
[36.8109, 128.6078],
[36.8183, 128.6046],
[36.8308, 128.6078],
[36.8674, 128.5253],
[36.8693, 128.5323],
[36.8731, 128.5242]
]

for i, (lat, lon) in enumerate(Shuttle_station):
    folium.CircleMarker(
        location=[lat, lon],
        radius=5, # Slightly larger radius for visibility
        color='black',
        fill=True,
        fill_color='black',
        fill_opacity=0.8,
        popup=f"User Point {i+1}: Lat {lat:.4f}, Lon {lon:.4f}"
    ).add_to(m_shuttle_clusters)

# Add LatLng popup functionality to the map
m_shuttle_clusters.add_child(folium.LatLngPopup())

# Display the map
display(m_shuttle_clusters)

Output hidden; open in https://colab.research.google.com to view.